![Week 6 Day 2](../../assets/banners/w6d2.png)

# Week 6 Day 2 - Solution: A Question Explorer Page

**The exercise:** write a new dashboard page that picks an exam, calls `GET /reports/{exam_id}`, draws a bar chart of each question's class mean and a table with the flagged counts, then test it headlessly with `AppTest` against a real API.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline in 🧪 demo mode.

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ No {key_name} set. That's fine this week: every cell runs offline in 🧪 demo mode.")
        print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import OUTPUT_DIR
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day2_solution"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

In [ ]:
# 🧱 Grade three synthetic students (approving every flag), so there are report cards to work with
n_students = min(3, len(datasets.synthetic_students()))  # 💡 3, or 2 with DATA_SIZE=small
batch_id = client.post(
    "/batches", data={"exam_id": "cs_midterm", "from_transcripts": "true", "students": str(n_students)}
).json()["batch_id"]
batch = wait_for(batch_id)
while batch["status"] == "paused":
    client.post(f"/batches/{batch_id}/review", json={"approved": True}).raise_for_status()
    batch = wait_for(batch_id)
print(f"{batch_id}: {batch['status']} · {[r['student_id'] for r in batch['reports']]}")

In [ ]:
import logging
import socket
import threading

import uvicorn


def start_api(app, port: int | None = None) -> tuple[str, uvicorn.Server]:
    """Serve `app` with uvicorn on a background thread, on 127.0.0.1 only. Returns (url, server)."""
    if port is None:
        with socket.socket() as s:  # 💡 ask the OS for a free port, so nothing clashes
            s.bind(("127.0.0.1", 0))
            port = s.getsockname()[1]
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning", access_log=False))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.05)
    return f"http://127.0.0.1:{port}", server


for name in ("uvicorn", "uvicorn.access", "uvicorn.error", "httpx", "streamlit"):
    logging.getLogger(name).setLevel(logging.ERROR)  # 💡 keep the notebook quiet

API_URL, server = start_api(app)
from gradewise.ui import client as ui  # noqa: E402

ui.API_URL = API_URL
print("🔌 API at", API_URL)

### The page
It reuses the shared pieces, so it looks and behaves like every other page: `styles.hero()`, `exam_picker()` (remembers the exam across pages), `friendly_errors()` (an API error becomes a red card), and `demo()` (the sidebar switch).

In [ ]:
PAGE = r"""
import streamlit as st

from gradewise.ui import client, styles
from gradewise.ui.common import demo, exam_picker, friendly_errors

styles.hero("Question explorer", "Which questions hurt, and which still have flags?", kicker="Your page")

exam_id = exam_picker()
if not exam_id:
    st.stop()
with friendly_errors():
    overview = client.get(f"/reports/{exam_id}", demo=demo())
if not overview["count"]:
    st.info("No report cards yet: grade a batch first.", icon="✍️")
    st.stop()

questions = overview["questions"]  # 💡 already sorted hardest first by the API
st.markdown(f"##### 🧗 Class mean per question ({overview['count']} students)")
ids, means = [q["question_id"] for q in questions], [q["mean_percent"] for q in questions]
styles.bar_chart(ids, means, max_value=100, suffix="%")
st.markdown("##### 🚩 Every question")
styles.table(
    ["Question", "Mean", "Mean %", "Flagged"],
    [(q["question_id"], f"{q['mean']:g}/{q['max_marks']:g}", f"{q['mean_percent']:g}", q["flagged"])
     for q in questions],
)
st.caption(f"Hardest: {questions[0]['question_id']} · flagged answers still open: {overview['flagged']}")
"""
page_path = SANDBOX / "question_explorer.py"
page_path.write_text(PAGE.strip() + "\n", encoding="utf-8")
print("📝 wrote", page_path)

### Test it headlessly

In [ ]:
import logging

from streamlit.testing.v1 import AppTest

for logger_name in list(logging.root.manager.loggerDict):  # 💡 keep Streamlit and uvicorn quiet in the notebook
    if logger_name.startswith(("streamlit", "uvicorn")):
        logging.getLogger(logger_name).setLevel(logging.ERROR)
at = AppTest.from_file(str(page_path), default_timeout=60)
at.session_state["demo"] = True
at.run()

print("Exceptions:", list(at.exception) or "none")
print("Exam picker:", at.selectbox[0].value)
print("Markdown   :", [m.value for m in at.markdown])
print("Caption    :", [c.value for c in at.caption])
assert not at.exception and at.caption[0].value.startswith("Hardest:")
server.should_exit = True

### Add it to the dashboard
Copy the file to `gradewise/ui/pages/question_explorer.py` **on your own branch**, and add one line to the `pages` list in `gradewise/ui/app.py`:

```python
st.Page(PAGES / "question_explorer.py", title="Question explorer", icon="🧗"),
```

Then `uv run python -m streamlit run gradewise/ui/app.py` and it's in the sidebar. 💡 Because the page only uses `client.get(...)`, it works unchanged against the API in Docker or in the cloud.